In [8]:
from __future__ import annotations

import json
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display
import plotly.graph_objects as go

# プロジェクトルートを sys.path に追加（notebook/LIST_Transcript/ の2つ上）
_project_root = Path().resolve().parents[1]
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from utils_core.config import ProjectConfig
config = ProjectConfig.from_env()
from research.ca_strategy_poc.scripts import run_buyhold_backtest_bloomberg as bt

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
ROOT = config.data_path.parent  # finance/
BBG_CSV = config.data_path / "Transcript" / "list_port_and_index_price_2015_2026.csv"
BBG_MCAP_JSON = config.data_path / "Transcript" / "list_port_mcap_2015_2026.json"
UNIVERSE_JSON = config.data_path / "Transcript" / "list_portfolio_20151224.json"
CONFIG_DIR = config.research_path / "ca_strategy_poc" / "config"
OUTPUT_DIR = (
    config.research_path / "ca_strategy_poc" / "workspaces" / "full_run" / "output"
)
RESULT_DIR = config.research_path / "ca_strategy_poc" / "reports"
FRED_CACHE_DIR = config.data_path / "raw" / "fred"

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
RISK_FREE_RATE_FALLBACK = 0.045  # annual fallback if FRED unavailable
RETURN_CAP = 0.50  # daily ±50%
START_DATE = date(2016, 1, 4)
END_DATE = date(2026, 2, 25)
TRADING_DAYS_PER_YEAR = 252

# Index column names in Bloomberg CSV
MSCI_KOKUSAI_COL = "MXKO INDEX"
MSCI_KOKUSAI_EW_COL = "MXKOEW INDEX"
MSCI_WORLD_COL = "MXWDJ INDEX"


## 1. データ確認


In [2]:
rf = bt.fetch_dgs10_daily_rf()
prices = bt.load_bloomberg_prices()
returns = bt.compute_daily_returns(prices=prices)
mcap = bt.load_bloomberg_mcap()

universe_data = bt.load_universe_data()
mapping = bt.build_bbg_ticker_map()


DGS10 loaded from cache: 2557 observations
Loading Bloomberg price data...
  Rows: 3169, Columns: 373
  Date range: 2015-12-24 ~ 2026-02-27
  Equity columns: 370
  Index columns: ['MXKO INDEX', 'MXKOEW INDEX', 'MXWDJ INDEX']
Loading Bloomberg market cap data...
  Rows: 720, Columns: 353
  Date range: 2014-12-31 ~ 2026-02-01


### Risk Free Rate確認


In [ ]:
display(rf)


2015-12-01    0.000084
2015-12-02    0.000086
2015-12-03    0.000091
2015-12-04    0.000089
2015-12-07    0.000088
                ...   
2026-02-18    0.000159
2026-02-19    0.000159
2026-02-20    0.000159
2026-02-23    0.000157
2026-02-24    0.000157
Name: DGS10, Length: 2557, dtype: float64

In [17]:
df_rf = pd.DataFrame(rf)

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=df_rf.index,y=df_rf['DGS10'], mode='lines')
    )
fig.update_layout(margin=dict(l=10, r=10, t=50, b=50))
fig.show()


### Universe data


In [3]:
df_universe_data = pd.DataFrame.from_dict(universe_data, orient='index')
df_universe_data = df_universe_data[0].apply(pd.Series)
df_universe_data.index.name='sedol'
df_universe_data = df_universe_data.reset_index()
display(df_universe_data)


,sedol,Name,Country,GICS_Sector,GICS_Industry,MSCI_Mkt_Cap_USD_MM,KY,AK,Total,Target_Weight,LIST,date,Bloomberg_Ticker,FIGI
0,0003128,Aberdeen Asset Management PLC,UNITED KINGDOM,Financials,Diversified Financials,3997.131,,,,,LIST,2015-12-24T00:00:00.000,ADN LN Equity,BBG000BB8XR2
1,0059585,ARM Holdings plc,UNITED KINGDOM,Information Technology,Semiconductors & Semiconductor Equipment,21780.040,,,,,LIST,2015-12-24T00:00:00.000,ARM LN Equity,BBG000C21L21
2,0141192,Sky plc,UNITED KINGDOM,Consumer Discretionary,Media,16953.160,,,,,LIST,2015-12-24T00:00:00.000,SKY LN Equity,BBG000DXH4L2
3,0233527,Croda International Plc,UNITED KINGDOM,Materials,Materials,6171.147,,,,,LIST,2015-12-24T00:00:00.000,CRDA NQ Equity,BBG002LCGPG5
4,0237400,Diageo plc,UNITED KINGDOM,Consumer Staples,Food Beverage & Tobacco,69771.760,,,,,LIST,2015-12-24T00:00:00.000,DGE LN Equity,BBG000BS69D5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403,BYVY8G0,Alphabet Inc. Class A,UNITED STATES,Information Technology,Software & Services,222006.500,3,2,5,0.85,LIST,2015-12-24T00:00:00.000,GOOGL US Equity,BBG009S39JX6
404,BYX4D52,HP Inc.,UNITED STATES,Information Technology,Technology Hardware & Equipment,21189.250,,,,,LIST,2015-12-24T00:00:00.000,HPQ US Equity,BBG000KHWT55
405,BYXW419,Mr Price Group Limited,SOUTH AFRICA,Consumer Discretionary,Retailing,3200.691,,,,,LIST,2015-12-24T00:00:00.000,MRP SJ Equity,BBG000BTPJN9
406,BYY88Y7,Alphabet Inc. Class C,UNITED STATES,Information Technology,Software & Services,231656.800,3,2,5,0.85,LIST,2015-12-24T00:00:00.000,GOOG US Equity,BBG009S3NB30


In [6]:
display(returns[['MXKO INDEX', 'MXKOEW INDEX']])


,MXKO INDEX,MXKOEW INDEX
Date,,
2015-12-25,0.000000,0.000000
2015-12-27,0.000000,0.000000
2015-12-28,-0.002626,-0.003465
2015-12-29,0.009340,0.007284
2015-12-30,-0.006102,-0.005779
...,...,...
2026-02-23,-0.009007,-0.008045
2026-02-24,0.005990,0.004907
2026-02-25,0.008473,0.004252


In [6]:
display(mcap[['WTB LN Equity']].dropna())


,WTB LN Equity
Date,
2015-02-26,14796.0636
2015-08-27,13426.3267
2016-03-03,9765.8662
2016-09-01,10163.2999
2017-03-02,8588.1284
2017-08-31,8886.7666
2018-03-01,9631.0177
2018-08-30,9606.8083
2019-02-28,11575.5057


In [7]:
mcap_ffill = mcap.ffill()
initial_mcap = pd.DataFrame(mcap_ffill.loc['2015-12-31'])
display(initial_mcap)
display(initial_mcap.dropna())


,2015-12-31
033780 KS Equity,11194.6352
035420 KS Equity,16340.3313
051900 KS Equity,13090.5417
090430 KS Equity,20595.2606
1044 HK Equity,11504.9473
...,...
WTB LN Equity,13426.3267
XLNX US Equity,10774.9624
XOM US Equity,323960.2000
YUM US Equity,31080.0000


,2015-12-31
033780 KS Equity,11194.6352
035420 KS Equity,16340.3313
051900 KS Equity,13090.5417
090430 KS Equity,20595.2606
1044 HK Equity,11504.9473
...,...
WTB LN Equity,13426.3267
XLNX US Equity,10774.9624
XOM US Equity,323960.2000
YUM US Equity,31080.0000
